In [1]:
import json
from datetime import datetime, timedelta

concepts_db = {}

def today():
    return datetime.now()

def serialize_date(dt):
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def deserialize_date(dt_str):
    return datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S")

def add_concept(name, subject, connections=[]):
    concepts_db[name] = {
        "subject": subject,
        "confidence": 0.5,
        "last_reviewed": None,
        "next_review": serialize_date(today()),
        "connections": connections
    }
    print(f"✅ Concept '{name}' added.")

def next_revision_days(confidence):
    if confidence < 0.3:
        return 1
    elif confidence < 0.6:
        return 3
    elif confidence < 0.8:
        return 7
    else:
        return 15

def update_concept(name, correct=True, time_taken=10):
    if name not in concepts_db:
        print("❌ Concept not found")
        return

    concept = concepts_db[name]
    confidence = concept["confidence"]

    if correct:
        if time_taken < 10:
            confidence += 0.15
            feedback = "⚡ Strong understanding"
        else:
            confidence += 0.05
            feedback = "🤔 Slow answer — possible weak clarity"
    else:
        confidence -= 0.2
        feedback = "❌ Weak concept — needs revision"

    confidence = max(0, min(1, confidence))

    concept["confidence"] = confidence
    concept["last_reviewed"] = serialize_date(today())

    days = next_revision_days(confidence)
    concept["next_review"] = serialize_date(today() + timedelta(days=days))

    print(f"\n📊 Updated: {name}")
    print(f"Confidence: {round(confidence,2)}")
    print(f"Next Review In: {days} days")
    print(f"Feedback: {feedback}")

def get_today_tasks():
    tasks = []
    now = today()

    for name, concept in concepts_db.items():
        next_review = deserialize_date(concept["next_review"])
        if next_review <= now:
            tasks.append((name, concept["confidence"]))

    tasks.sort(key=lambda x: x[1])

    print("\n📅 Today's Revision Tasks:")
    if not tasks:
        print("🎉 Nothing to revise today!")
    else:
        for t in tasks:
            print(f"🔹 {t[0]} (Confidence: {round(t[1],2)})")

def check_risk():
    print("\n⚠️ At-Risk Concepts:")
    found = False
    for name, concept in concepts_db.items():
        if concept["confidence"] < 0.4:
            print(f"🚨 {name} — You are about to forget this!")
            found = True
    if not found:
        print("✅ No high-risk concepts")

def show_all():
    print("\n📚 All Concepts:")
    for name, c in concepts_db.items():
        print(f"{name} | Conf: {round(c['confidence'],2)} | Next: {c['next_review']}")

add_concept("Cardiac Cycle", "Biology", ["Blood Pressure", "Heart Rate"])
add_concept("Nernst Equation", "Chemistry", ["Electrochemistry"])
add_concept("Projectile Motion", "Physics", ["Kinematics"])

update_concept("Cardiac Cycle", correct=True, time_taken=8)
update_concept("Nernst Equation", correct=False)
update_concept("Projectile Motion", correct=True, time_taken=15)

get_today_tasks()
check_risk()
show_all()

✅ Concept 'Cardiac Cycle' added.
✅ Concept 'Nernst Equation' added.
✅ Concept 'Projectile Motion' added.

📊 Updated: Cardiac Cycle
Confidence: 0.65
Next Review In: 7 days
Feedback: ⚡ Strong understanding

📊 Updated: Nernst Equation
Confidence: 0.3
Next Review In: 3 days
Feedback: ❌ Weak concept — needs revision

📊 Updated: Projectile Motion
Confidence: 0.55
Next Review In: 3 days
Feedback: 🤔 Slow answer — possible weak clarity

📅 Today's Revision Tasks:
🎉 Nothing to revise today!

⚠️ At-Risk Concepts:
🚨 Nernst Equation — You are about to forget this!

📚 All Concepts:
Cardiac Cycle | Conf: 0.65 | Next: 2026-03-29 16:46:43
Nernst Equation | Conf: 0.3 | Next: 2026-03-25 16:46:43
Projectile Motion | Conf: 0.55 | Next: 2026-03-25 16:46:43


In [2]:
get_today_tasks()


📅 Today's Revision Tasks:
🎉 Nothing to revise today!


In [3]:
check_risk()


⚠️ At-Risk Concepts:
🚨 Nernst Equation — You are about to forget this!


In [4]:
!pip install streamlit pyngrok networkx matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 76.2 MB/s eta 0:00:00


In [15]:
%%writefile app.py
import streamlit as st
from datetime import datetime, timedelta
import networkx as nx
import matplotlib.pyplot as plt

# ------------------------------
# INIT
# ------------------------------
st.set_page_config(layout="wide")
if "concepts_db" not in st.session_state:
    st.session_state.concepts_db = {}

def today():
    return datetime.now()

def next_revision_days(conf):
    if conf < 0.3: return 1
    elif conf < 0.6: return 3
    elif conf < 0.8: return 7
    else: return 15

# ------------------------------
# ADD CONCEPT
# ------------------------------
def add_concept(name, subject, connections):
    st.session_state.concepts_db[name] = {
        "subject": subject,
        "confidence": 0.5,
        "last_reviewed": None,
        "next_review": today(),
        "connections": connections,
        "history": []
    }

# ------------------------------
# UPDATE
# ------------------------------
def update_concept(name, correct, time_taken):
    c = st.session_state.concepts_db[name]
    conf = c["confidence"]

    if correct:
        if time_taken < 10:
            conf += 0.15
            feedback = "⚡ Strong"
        else:
            conf += 0.05
            feedback = "🤔 Slow"
    else:
        conf -= 0.2
        feedback = "❌ Weak"

    conf = max(0, min(1, conf))
    c["confidence"] = conf
    c["last_reviewed"] = today()
    c["next_review"] = today() + timedelta(days=next_revision_days(conf))
    c["history"].append(conf)

    return feedback

# ------------------------------
# UI HEADER
# ------------------------------
st.title("🧠 AI Cognitive Learning System")

col1, col2, col3 = st.columns(3)

total = len(st.session_state.concepts_db)
weak = sum(1 for c in st.session_state.concepts_db.values() if c["confidence"] < 0.4)
strong = sum(1 for c in st.session_state.concepts_db.values() if c["confidence"] > 0.75)

col1.metric("📚 Total Concepts", total)
col2.metric("⚠️ Weak Concepts", weak)
col3.metric("💪 Strong Concepts", strong)

# ------------------------------
# ADD
# ------------------------------
st.sidebar.header("➕ Add Concept")
name = st.sidebar.text_input("Concept")
subject = st.sidebar.selectbox("Subject", ["Biology","Physics","Chemistry"])
connections = st.sidebar.text_input("Connections (comma separated)")

if st.sidebar.button("Add"):
    add_concept(name, subject, connections.split(","))
    st.sidebar.success("Added!")

# ------------------------------
# PRACTICE
# ------------------------------
st.header("📊 Practice Simulator")

if st.session_state.concepts_db:
    selected = st.selectbox("Select Concept", list(st.session_state.concepts_db.keys()))
    correct = st.radio("Correct?", ["Yes","No"]) == "Yes"
    time_taken = st.slider("Time Taken", 1, 30, 10)

    if st.button("Submit"):
        fb = update_concept(selected, correct, time_taken)
        st.success(fb)

        # Illusion detection
        if correct and time_taken > 15:
            st.warning("🧠 Illusion detected: You answered correctly but slowly!")

# ------------------------------
# AI COACH
# ------------------------------
st.header("🤖 AI Coach Insights")

for name, c in st.session_state.concepts_db.items():
    if c["confidence"] < 0.4:
        st.error(f"🚨 Focus NOW: {name}")
    elif c["confidence"] > 0.8:
        st.info(f"✅ Stop over-revising: {name}")

# ------------------------------
# REVISION
# ------------------------------
st.header("📅 Today's Tasks")

for name, c in st.session_state.concepts_db.items():
    if c["next_review"] <= today():
        hours_left = (c["next_review"] - today()).total_seconds()/3600
        st.warning(f"{name} → revise NOW!")

# ------------------------------
# FORGETTING PREDICTION
# ------------------------------
st.header("🔮 Forgetting Prediction")

for name, c in st.session_state.concepts_db.items():
    time_left = c["next_review"] - today()
    hours = int(time_left.total_seconds()/3600)

    if hours > 0:
        st.write(f"{name}: ⏳ {hours} hrs left before forgetting")

# ------------------------------
# GRAPH
# ------------------------------
st.header("🧠 Knowledge Graph")

G = nx.Graph()

for name, c in st.session_state.concepts_db.items():
    conf = c["confidence"]

    # color logic
    if conf < 0.4:
        color = "red"
    elif conf < 0.75:
        color = "yellow"
    else:
        color = "green"

    G.add_node(name, color=color, size=conf*1000)

    for conn in c["connections"]:
        if conn.strip():
            G.add_edge(name, conn.strip())

if G.nodes:
    pos = nx.spring_layout(G)
    colors = [G.nodes[n]['color'] for n in G.nodes]
    sizes = [G.nodes[n]['size'] for n in G.nodes]

    plt.figure(figsize=(10,7))
    nx.draw(G, pos, with_labels=True, node_color=colors, node_size=sizes)
    st.pyplot(plt)

# ------------------------------
# TREND GRAPH
# ------------------------------
st.header("📈 Learning Trend")

for name, c in st.session_state.concepts_db.items():
    if c["history"]:
        st.write(f"### {name}")
        st.line_chart(c["history"])

Overwriting app.py


In [18]:
!ngrok config add-364XD3cT2wVnw8OsJ1LLh8ortXK_4LmyuGZGUGxFZ9g7RcGqf
from pyngrok import ngrok

!streamlit run app.py &>/dev/null &

public_url = ngrok.connect(8501)
public_url

NAME:
  config - update or migrate ngrok's configuration file

USAGE:
  ngrok config [flags]

DESCRIPTION: 
  The config command gives a quick way to create or update ngrok's configuration
  file. Use 'add-authtoken' or 'add-api-key' to set the corresponding properties.

  Use 'check' to test a configuration file for validity. If you have an old
  configuration file, you can also use 'upgrade' to automatically migrate to the
  latest version.

COMMANDS:
  add-api-key                    save api key to configuration file
  add-authtoken                  save authtoken to configuration file
  add-connect-url                adds the connect URL (connect_url) to configuration file for custom agent ingress
  add-server-addr                alias of add-connect-url
  check                          check configuration file
  edit                           edit configuration file
  upgrade                        auto-upgrade configuration file

OPTIONS:
      --config strings   path to config f

<NgrokTunnel: "https://nona-cleverish-ava.ngrok-free.dev" -> "http://localhost:8501">